In [2]:
!pip install cvxopt
!pip install pandas
!pip install numpy

In [3]:
import pandas as pd
import numpy as np
from cvxopt import matrix as matrix
from cvxopt import solvers as solvers

In [34]:
def kernel_poly2(x1, x2, d):
  kernel = (x1.T @ x2 + d)**2
  return kernel.item()

def calculate_bias(supp_v, supp_v_alp, supp_v_class, kernel):
  tot_bias = 0
  w0 = 0
  for i in range(len(supp_v)):
    decision_value = 0
    for j in range(len(supp_v)):
      k_input1 = supp_v[i].reshape(-1,1)
      k_input2 = supp_v[j].reshape(-1,1)
      decision_value += (supp_v_alp[j] * supp_v_class[j] * kernel(k_input1, k_input2, 1))
    tot_bias += (supp_v_class[i] - decision_value)
  w0 = tot_bias / len(supp_v)
  return w0

# read the data
data = pd.read_csv('t030.csv', header=None) # two features, two classes
features = data.iloc[:, 0:2]
classes = data.iloc[:, 2]
# polynomial kernel
feat1 = np.array(features.loc[:,0])
feat2 = np.array(features.loc[:,1])
X = np.column_stack((feat1, feat2))
y = np.where(classes == 1, -1, 1)
bias = 1
# kernel matrix
N = X.shape[0]
K = np.zeros((N, N))
for i in range(N):
  for j in range(N):
    K[i, j] = kernel_poly2(X[i].reshape(-1,1), X[j].reshape(-1,1), bias)
# quadratic programming
P = matrix(np.outer(y, y) * K)
q = matrix(-np.ones((N, 1)))
G = matrix(np.vstack((-np.eye(N), np.eye(N))))
h = matrix(np.hstack((np.zeros(N), np.ones(N)*10))) # C = 10
A = matrix(y.reshape(1,-1).astype(float))
b = matrix(0.0)
# solver parametrs
solvers.options['show_progress'] = False
solvers.options['abstol'] = 1e-10
solvers.options['reltol'] = 1e-10
solvers.options['feastol'] = 1e-10
# run solver
sol = solvers.qp(P, q, G, h, A, b)
alphas = np.array(sol['x']).flatten()
# Support vectors based on close-to-zero values and discriminant function parameters
sv = alphas > 1e-8
# the support vectors
sv_indices = (sv)
sum_sv_indices = np.sum(sv_indices)
# Make sure sv_indices is the right shape
sv_threshold = 1e-8
sv_indices = (alphas > sv_threshold)
supp_vec = X[sv_indices]
supp_labels = y[sv_indices]
print(f"Number of support vectors:{len(supp_vec)}")
print(f"Class 1: {np.sum(supp_labels == -1)}")
print(f"Class 2: {np.sum(supp_labels == 1)}")
# solve w0
w0 = calculate_bias(supp_vec, alphas[sv_indices], supp_labels, kernel_poly2)

Number of support vectors:4
Class 1: 2
Class 2: 2


In [35]:
def svm_prediction(x,sup_v,sup_v_a, sup_v_lab, w0, kernel, bias):
  x = np.array(x)
  # print(f"NUmber of elements in x: {len(x)}")
  print(f"NUmber of elements in sup_v: {len(sup_v)}")
  print(f"NUmber of elements in sup_v_lab: {len(sup_v_lab)}")
  preds = []
  for i in range(len(x)):
    x_x = x[i]
    print(f"Processing sample {i}: {x_x}")
    decision_value = w0
    # sum all the support vectors
    for j in range(len(sup_v)):
      decision_value += (sup_v_a[j] * sup_v_lab[j] * kernel(x_x.reshape(-1,1), sup_v[j].reshape(-1,1),bias))
    if decision_value >= 0:
      preds.append(1)
    else:
      preds.append(-1)
  # ensure it is an array
  predictions = np.array(preds)
  print(f"Number of predictions: {len(predictions)}")
  return predictions

# split to training and testing
split_ratio = 0.80
train_data = X[:int(split_ratio*len(X)),:]
train_class = y[:int(split_ratio*len(y))]
test_data = X[int(split_ratio*len(X)):,:]
test_class = y[int(split_ratio*len(y)):]
print(np.unique(test_class))
# now use n-svm on train data and train class
bias = 1
# kernel matrix
N = train_data.shape[0]
K = np.zeros((N, N))
for i in range(N):
  for j in range(N):
    K[i, j] = kernel_poly2(train_data[i].reshape(-1,1), train_data[j].reshape(-1,1), bias)
# quadratic programming
P = matrix(np.outer(train_class, train_class) * K)
q = matrix(-np.ones((N, 1)))
G = matrix(np.vstack((-np.eye(N), np.eye(N))))
h = matrix(np.hstack((np.zeros(N), np.ones(N)*10))) # C = 10
A = matrix(train_class.reshape(1,-1).astype(float))
b = matrix(0.0)
# solver parametrs
solvers.options['show_progress'] = False
solvers.options['abstol'] = 1e-10
solvers.options['reltol'] = 1e-10
solvers.options['feastol'] = 1e-10
# run solver
sol = solvers.qp(P, q, G, h, A, b)
alphas = np.array(sol['x']).flatten()
# Support vectors based on close-to-zero values and discriminant function parameters
sv = alphas > 1e-8
# the support vectors
sv_indices = (sv)
sum_sv_indices = np.sum(sv_indices)
# Make sure sv_indices is the right shape
sv_threshold = 1e-8
sv_indices = (alphas > sv_threshold)
supp_vec = train_data[sv_indices]
supp_labels = train_class[sv_indices]
# now use the svm
y_pred = svm_prediction(test_data, supp_vec, alphas[sv_indices], supp_labels, w0, kernel_poly2, 1)
# calculate accuracy
accuracy = np.mean(y_pred.flatten() == test_class)
print(f"Overall accuracy is: {accuracy}")

[-1  1]
NUmber of elements in sup_v: 2
NUmber of elements in sup_v_lab: 2
Processing sample 0: [13.20837216 13.99195318]
Processing sample 1: [12.17513019 14.90570126]
Processing sample 2: [10.47978899 12.63741199]
Processing sample 3: [12.72822912 11.42171378]
Processing sample 4: [11.8540133  10.32346301]
Processing sample 5: [12.72404532 14.18187802]
Processing sample 6: [10.72660996 10.85760128]
Processing sample 7: [14.  7.]
Processing sample 8: [20. 17.]
Number of predictions: 9
Overall accuracy is: 0.8888888888888888
